#**Part 1 — Python & Data Foundations: ingest, validate, clean, enrich**

##Task 1.1 — Read the file the foundations way

In [38]:
import csv
import json

In [39]:
source_path = "/content/appointments.csv"

In [40]:
accepted = []
rejected = []

In [41]:
def parse_row(row):
  converted = row.copy()

  #convert age, fee_inr, and prior_no_shows to int
  #pattern : converted["age"] = int(row["age"])

  #convert distance_km to float

  converted['age']= int(row['age'])
  converted['fee_inr'] = int(row['fee_inr'])
  converted['prior_no_shows'] = int(row['prior_no_shows'])
  converted['distance_km'] = float(row['distance_km'])


  return converted


with open(source_path, "r", newline="", encoding="utf-8") as file:
  reader = csv.DictReader(file)
  print("Header:", reader.fieldnames)
  rows = list(reader)

print("Row count:", len(rows))

for row in rows:
  try:
    converted = parse_row(row)
  except ValueError:
    rejected.append(row)
  else:
    accepted.append(converted)

with open("rejected_rows.json","w",encoding="utf-8") as file:
  json.dump(rejected, file, indent=2)

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020
Accepted: 988
Rejected: 32


##Task 1.2 — Inspect and clean, pass 1


In [42]:
import pandas as pd

In [43]:
df = pd.read_csv("/content/appointments.csv")

print("Raw shape:", df.shape)

df.info()

print("\nMissing values:\n", df.isnull().sum())
print("\nExact duplicates:", df.duplicated().sum())

# 1. Remove exact duplicate rows
df = df.drop_duplicates()
print("\nRows after dropping duplicates:", len(df))

# 2. Normalize booking channel labels
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
print("\nRows after cleaning booking_channel:", len(df))
print("\nBooking channel labels:", sorted(df["booking_channel"].dropna().unique()))

# 3. Convert the date columns
df['appt_date'] = pd.to_datetime(df['appt_date'], errors="coerce")
df['booked_on'] = pd.to_datetime(df['booked_on'])
print("\nRows after date conversion:", len(df))

# 4. Remove rows with a missing appointment date or status
df = df.dropna(subset=["appt_date","status"])
print("\nRows after dropping missing appt_date/status:", len(df))



Raw shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
 appointment_id      0
patient_id          0
clinic_i

##Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [44]:
import numpy as np

In [45]:
# Mark invalid ages as missing, then calculate and apply the median.
invalid_age_mask = (df['age'] < 1) | (df['age'] > 100)
invalid_age_count = int(invalid_age_mask.sum())
df.loc[invalid_age_mask, "age"] = np.nan

# Fill the gaps after invalid ages have been marked.
age_median = df['age'].median()
distance_median = df['distance_km'].median()

df['age'] = df['age'].fillna(age_median).round().astype(int)

df['distance_km'] = df['distance_km'].fillna(distance_median)

#calculate distance fences after filling gaps, then cap outliers

distance_q1 = df['distance_km'].quantile(0.25)
distance_q3 = df['distance_km'].quantile(0.75)

distance_iqr = distance_q3 - distance_q1

distance_lower = distance_q1 - 1.5 * distance_iqr

distance_upper = distance_q3 + 1.5 * distance_iqr

distance_outlier_mask = (
    (df['distance_km'] < distance_lower) | (df['distance_km'] > distance_upper)
)

distance_capped_count = int(distance_outlier_mask.sum())

df['distance_km'] = np.clip(
    df['distance_km'], distance_lower, distance_upper
)

#calculate fee fences and cap ouliers

fee_q1 = df['fee_inr'].quantile(0.25)
fee_q3 = df['fee_inr'].quantile(0.75)
fee_iqr = fee_q3 - fee_q1

fee_lower = fee_q1 - 1.5 * fee_iqr
fee_upper = fee_q3 + 1.5 * fee_iqr

fee_outlier_mask = (df['fee_inr'] < fee_lower) | (df['fee_inr'] > fee_upper)

fee_capped_count = int(fee_outlier_mask.sum())

df['fee_inr'] = np.clip(df['fee_inr'], fee_lower, fee_upper)

#Apply final rounding
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

print("Invalid ages:", invalid_age_count)
print("Age median used:", age_median)
print("Distance median used:", distance_median)
print("Distance values capped:", distance_capped_count)
print("Fee values capped:", fee_capped_count)

Invalid ages: 5
Age median used: 48.0
Distance median used: 8.25
Distance values capped: 26
Fee values capped: 3


##Task 1.4 — Features and a NumPy summary

In [46]:
#Lead time and target
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = df['status'].eq("No-Show").astype(int)

#Numeric matrix in the requested column order
feature_columns = ['age', 'distance_km', 'fee_inr', 'lead_days']
X = df[feature_columns].to_numpy()

# Mean of each column
print("Means:", np.round(X.mean(axis=0),2))

#Count and share booked more than 14 days ahead
long_lead_mask = df['lead_days'] > 14
print("\nLead days > 14:", int(long_lead_mask.sum()),f"({long_lead_mask.mean()*100:.1f}%)")

#Distance z-scores using NumPy's default population standard deviation

distance = X[:, 1]
z_scores = (distance - np.mean(distance))/np.std(distance)
print("\nDistance rows with |z|>2:", int((np.abs(z_scores) > 2).sum()))

print("Lead day range:", df["lead_days"].min(),"to",df["lead_days"].max())

print("\nNo-shows:", int(df["no_show"].sum()),"of", len(df))


Means: [ 48.17   9.5  635.71  11.82]

Lead days > 14: 296 (29.9%)

Distance rows with |z|>2: 53
Lead day range: 1 to 45

No-shows: 218 of 990


##Task 1.5 — Enrich the clinics from a live API

In [47]:
import requests

In [48]:
def fetch_temp(latitude, longitude):
  params = {
      "latitude": latitude,
      "longitude": longitude,
      "hourly": "temperature_2m",
      "forecast_days": 1
  }

  reason = "No successful response"

  for attempt in range(2): #first attempt plus one entry
    try:
      response = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout = 10)

      if response.status_code != 200:
        reason = f"HTTP status {response.status_code}"
        continue

      data = response.json()

      if not isinstance(data, dict):
          reason = "Response was not a JSON object"
          continue

      hourly = data.get("hourly")
      temperatures = (
          hourly.get("temperature_2m")
          if isinstance(hourly, dict)
          else None
      )

      if not isinstance(temperatures, list) or len(temperatures) == 0:
        reason = "Response is missing hourly.temperature_2m values"
        continue

      temp_c = temperatures[0]

      if temp_c is None:
          reason = "First hourly temperature value is missing"
          continue

      return temp_c, None

    except requests.exceptions.RequestException as exc:
      reason = f"Request error: {exc}"
    except ValueError as exc:
      reason = f"Invalid JSON response: {exc}"

  return None, reason

In [49]:
clinics = pd.read_csv("/content/clinics.csv")


temperatures = []
failures = []

for _, clinic in clinics.iterrows():
  temp_c, reason = fetch_temp( clinic['latitude'], clinic['longitude'])
  temperatures.append(temp_c)

  if temp_c is None:
    message = f"{clinic['clinic_name']}: {reason}"
    print(message)
    failures.append(message)


clinics["temp_c"] = temperatures
clinics.to_csv("clinics_enriched.csv", index=False)

print("\nRow count:", len(clinics))
print("\ntemp_c values:", clinics["temp_c"].tolist())
print("\nFailure count:", len(failures))


Row count: 8

temp_c values: [14.4, 15.7, 22.3, 15.3, 15.3, 24.2, 15.2, 17.0]

Failure count: 0


##Task 1.6 — Export the clean table and write the SQL load script   

In [50]:
export_df = df.sort_values("appointment_id").copy()

for date_col in ["booked_on", "appt_date"]:
  export_df[date_col] = export_df[date_col].dt.strftime("%Y-%m-%d")

export_columns = [
     "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
 ]


export_df = export_df[export_columns]

print("Appointment IDs are unique:", export_df["appointment_id"].is_unique)
print("Total missing values:", int(export_df.isna().sum().sum()))

export_df.to_csv("appointments_clean.csv", index=False)
print("Saved rows:", len(export_df))

Appointment IDs are unique: True
Total missing values: 0
Saved rows: 990


In [51]:
clinics_df = pd.read_csv("/content/clinics_enriched.csv")

In [52]:
sql_lines = []

# Write clinic inserts first so appointment foreign keys can be loaded.
for _, clinic in clinics_df.iterrows():
    latitude = (
        "NULL" if pd.isna(clinic["latitude"])
        else f"{float(clinic['latitude']):.4f}"
    )
    longitude = (
        "NULL" if pd.isna(clinic["longitude"])
        else f"{float(clinic['longitude']):.4f}"
    )
    temp_c = (
        "NULL" if pd.isna(clinic["temp_c"])
        else str(float(clinic["temp_c"]))
    )

    sql_lines.append(
        f"INSERT INTO clinics VALUES "
        f"('{clinic['clinic_id']}', '{clinic['clinic_name']}', "
        f"'{clinic['city']}', '{clinic['tier']}', "
        f"{latitude}, {longitude}, {temp_c});"
    )

# Append one appointment insert for every row, in table column order.
for _, row in export_df.iterrows():
    sql_lines.append(
        f"INSERT INTO appointments VALUES "
        f"('{row['appointment_id']}', '{row['patient_id']}', "
        f"'{row['clinic_id']}', '{row['booked_on']}', "
        f"'{row['appt_date']}', {int(row['age'])}, "
        f"'{row['gender']}', {float(row['distance_km']):.1f}, "
        f"{int(row['fee_inr'])}, '{row['booking_channel']}', "
        f"'{row['visit_type']}', '{row['reminder_sent']}', "
        f"{int(row['prior_no_shows'])}, '{row['status']}', "
        f"{int(row['lead_days'])}, {int(row['no_show'])});"
    )

with open("load_data.sql", "w", encoding="utf-8") as file:
    file.write("\n".join(sql_lines) + "\n")

# Verify statement count and order
insert_lines = [
    line for line in sql_lines
    if line.upper().startswith("INSERT INTO")
]

print("Total INSERT statements:", len(insert_lines))
print("First statement:", insert_lines[0])
print("Ninth statement:", insert_lines[8])

assert len(clinics_df) == 8
assert len(export_df) == 990
assert len(insert_lines) == 998
assert all(line.upper().startswith("INSERT INTO CLINICS") for line in insert_lines[:8])
assert insert_lines[8].upper().startswith("INSERT INTO APPOINTMENTS")

Total INSERT statements: 998
First statement: INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 14.4);
Ninth statement: INSERT INTO appointments VALUES ('A0001', 'P509', 'C05', '2025-01-24', '2025-01-25', 60, 'F', 18.0, 850, 'Phone', 'Consultation', 'Yes', 1, 'No-Show', 1, 1);


#**Part 2 — SQL: verify the numbers and find where no-shows cluster**

##Task 2.1 — Build and load the database


In [53]:
from google.colab import files
files.download("load_data.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Task 2.2 — No-show rate by clinic

In [54]:
appointments = pd.read_csv("/content/appointments_clean.csv")
clinics = pd.read_csv("/content/clinics_enriched.csv")

clinic_summary = (
    appointments.groupby("clinic_id")
    .agg(
        appointments = ("appointment_id","size"),
        no_shows=("no_show","sum")
    )
    .reset_index()
)

clinic_summary["no_show_pct"] = (
    100 * clinic_summary["no_shows"] / clinic_summary["appointments"]
).round(1)


clinic_summary = clinic_summary.merge(
    clinics[["clinic_id","clinic_name","city"]],
    on="clinic_id",
    how="left"
)

clinic_summary = clinic_summary[
    ["clinic_name","city", "appointments", "no_shows", "no_show_pct"]
].sort_values("no_show_pct", ascending=False)

print(clinic_summary.to_string(index=False))

             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
Edappally Medical Centre   Kochi           142        23         16.2
  Sitabuldi Health Point  Nagpur            82        13         15.9
   Vaishali Nagar Clinic  Jaipur           130        18         13.8
